In [ ]:
# Passo a passo
# Passo 1: Abrir o navegador
# Passo 2: Carregar a lista de produtos (livros)
# Passo 3: Para cada item da lista (para cada livro)
    # Passo 4: Pesquisar no gutenberg: https://www.gutenberg.org/
    # Passo 5: Se não encontrar:
        # Pesquisar no https://books.toscrape.com/
    # Passo 6: Registrar preço e link do livro na planilha

In [ ]:

#pip install selenium
#pip install webdriver-manager
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

options = webdriver.ChromeOptions()
# Como você quer usar um perfil temporário/isolado, não precisamos de user-data-dir.

# O Selenium gerencia o driver automaticamente agora, basta iniciar direto:
navegador = webdriver.Chrome(options=options)
# Passo a passo
# Passo 1: Abrir o navegador
# criar o navegador


In [ ]:
#Passo 2: Carregar a lista de produtos (livros)
#pip install pandas
#pip install openpyxl
import pandas as pd
df_produtos = pd.read_excel('Produtos.xlsx')
display(df_produtos)

In [ ]:
#Passo 4: Pesquisar no gutenberg: https://www.gutenberg.org/
def pesquisar_gutenberg(nome_livro, autor, navegador):
    lista_palavras_autor = autor.split(" ")
    navegador.get('https://www.gutenberg.org/')
    

    #navegador.find_element(quem procurar, como procurar)
    elemento = navegador.find_element('class name', 'search-input')
    elemento.send_keys(nome_livro)
    elemento.send_keys(Keys.ENTER)
    #passo 5: ver se encontrou ou não encontrar:
    lista_resultados = navegador.find_elements('class name', 'booklink')
    #print(lista_resultados)
    link = None

    for resultado in lista_resultados:
        texto = resultado.text    
        #print(nome_livro.lower())
        #print(autor.lower())
        if (nome_livro.lower() in texto.lower()) and (all(palavra.lower() in autor.lower() for palavra in lista_palavras_autor)):
            #print(texto)
            link = resultado.find_element('class name', 'link').get_attribute('href')
            #print(link)
            break
    if link:
        preco = 0   
    else:
        preco = None 

    return (link, preco)

#passo 5 se não encontrar
#Pesquisar no https://books.toscrape.com/
def pesquisar_toscrape(nome_livro, autor, categoria, navegador):
    navegador.get('https://books.toscrape.com/')
    #pegar toda a barra lateral das categorias
    lista_categorias = navegador.find_element('class name', 'nav-list')

    try:
        lista_categorias.find_element('link text', categoria).click()    
    except:
        print('Nenhuma categoria selecionada')

    
    link = None
    preco = None
    encontrado = False
    while not encontrado:
        #olhar todos os resultados da pagina
        lista_resultados = navegador.find_elements('class name', 'product_pod')    
        for resultado in lista_resultados:
            elemento = resultado.find_element('tag name', 'h3')
            elemento_do_link = elemento.find_element('tag name', 'a')
            titulo = elemento_do_link.get_attribute('title')
            if nome_livro.lower() in titulo.lower():
                link = elemento_do_link.get_attribute('href')
                elemento_preço = resultado.find_element('class name', 'price_color')
                preco = elemento_preço.text
                encontrado = True
                break

        #passar para a proxima pagina    
        #tentar clicar no botão next
        try:
            navegador.find_element('link text', 'next' ).click()
        except:
            break

        #se não existir interompe o while
    return(link, preco)




In [ ]:

# nome_livro = 'Frankenstein'
# autor = 'Mary Shelley'
# categorias = 'Classics'
# link, preco =  pesquisar_gitenberg(nome_livro, autor, navegador)
# print(link, preco)

# #Passo 3: Para cada item da lista (para cada livro)

# nome_livro = 'Sapiens'
# autor = 'Yuval Harari'
# categorias = 'History'
# link, preco = pesquisar_toscrape(nome_livro, autor, categorias, navegador)
# print(link,preco)

# Converte as colunas para o tipo object (texto) para aceitar links e preços formatados
df_produtos['link'] = df_produtos['link'].astype(object)
df_produtos['preco'] = df_produtos['preco'].astype(object)


for linha in df_produtos.index:    
    nome_livro = df_produtos.loc[linha, 'nome']
    autor = df_produtos.loc[linha, 'autor']
    categoria = df_produtos.loc[linha, 'categoria']    

    link1, preco1 = pesquisar_gutenberg(nome_livro, autor, navegador)
    print(nome_livro, link1, preco1)
    if not link1:
        link2, preco2 = pesquisar_toscrape(nome_livro, autor, categoria, navegador)
        print(nome_livro, link2, preco2)

        #Passo 6: Registrar preço e link do livro na planilha
        df_produtos.loc[linha, 'preco'] = preco2
        df_produtos.loc[linha, 'link'] = link2
    else:
        df_produtos.loc[linha, 'preco'] = preco1
        df_produtos.loc[linha, 'link'] = link1 

In [ ]:
#Passo 7: Salvar a planilha com os preços e links encontrados
df_produtos.to_excel("Produtos_atualizados.xlsx", index=False)

#Passos 8: Fechar o navegador
navegador.quit()

#Passos 9: Finalizar o programa